In [34]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, RepeatedKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)

In [3]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [8]:
target_map = {
    "Presence": 1,
    "Absence": 0
}

train_df["Heart Disease"] = train_df["Heart Disease"].map(target_map)

In [4]:
train_df['HR_cut'] = pd.qcut(train_df['Max HR'], 8)
test_df['HR_cut'] = pd.qcut(test_df['Max HR'], 8)

bins = [70.999, 126, 142, 150, 157, 162, 166, 173, 202]
labels = [0, 1, 2, 3, 4, 5, 6, 7]

train_df['HR_group'] = pd.cut(train_df['Max HR'], bins=bins, labels=labels, right=True)
test_df['HR_group']  = pd.cut(test_df['Max HR'], bins=bins, labels=labels, right=True)

In [5]:
train_df['cholesterol_cut'] = pd.qcut(train_df['Cholesterol'], 8)
test_df['cholesterol_cut'] = pd.qcut(test_df['Cholesterol'], 8)

bins = [125, 204, 223, 233, 243, 254, 269, 286, 564]
labels = [0, 1, 2, 3, 4, 5, 6, 7]

train_df['cholesterol_group'] = pd.cut(train_df['Cholesterol'], bins=bins, labels=labels, right=True)
test_df['cholesterol_group']  = pd.cut(test_df['Cholesterol'], bins=bins, labels=labels, right=True)

In [26]:
X = train_df[[
    "Thallium",
    "Chest pain type",
    "Exercise angina",
    "Number of vessels fluro",
    "ST depression",
    "Slope of ST",
    "Max HR"
]]

y = train_df["Heart Disease"]

model = LinearRegression()
model.fit(X, y)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [27]:
coefs = model.coef_
intercept = model.intercept_ 

train_df["reg_feature"] = train_df["Thallium"] * coefs[0] + \
train_df["Chest pain type"] * coefs[1] + \
train_df["Exercise angina"] * coefs[2] + \
train_df["Number of vessels fluro"] * coefs[3] + \
train_df["ST depression"] * coefs[4] + \
train_df["Slope of ST"] * coefs[5] + \
train_df["Max HR"] * coefs[6] + \
intercept

test_df["reg_feature"] = test_df["Thallium"] * coefs[0] + \
                         test_df["Chest pain type"] * coefs[1] + \
                         test_df["Exercise angina"] * coefs[2] + \
                         test_df["Number of vessels fluro"] * coefs[3] + \
                         test_df["ST depression"] * coefs[4] + \
                         test_df["Slope of ST"] * coefs[5] + \
                         test_df["Max HR"] * coefs[6] + \
                         intercept

In [13]:
bins = [-float('inf'), 0, 1, 2, 3, float('inf')]
labels = [0, 1, 2, 3, 4]

train_df['ST_depression_group'] = pd.cut(train_df["ST depression"], bins=bins, labels=labels)
test_df['ST_depression_group'] = pd.cut(test_df["ST depression"], bins=bins, labels=labels)

In [14]:
cols_to_encode = ["Thallium", 
                  "Chest pain type", 
                  "Exercise angina", 
                  "Number of vessels fluro",
                  "Slope of ST",
                  "HR_group",
                  "ST_depression_group"]

In [15]:
def train_target_encoding(cols_to_encode):
    results = []
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    for col in cols_to_encode:
        train_df[f"tar_{col}"] = np.nan
    
    for train_idx, val_idx in kf.split(train_df):
        df_train_split = train_df.iloc[train_idx].copy()
        df_val_split = train_df.iloc[val_idx].copy()
        
        for col in cols_to_encode:
            mean = df_train_split.groupby(col)["Heart Disease"].mean()
            
            df_val_split[f"tar_{col}"] = df_val_split[col].map(mean).astype(float)
            
        results.append(df_val_split)
    
    encoded_df = pd.concat(results).sort_index()

    return encoded_df

In [16]:
def test_target_encoding(df, cols_to_encode):
    
    results = []
      
    for col in cols_to_encode:
        df[f"tar_{col}"] = np.nan
        mean = train_df.groupby(col)["Heart Disease"].mean()
        df[f"tar_{col}"] = df[col].map(mean).astype(float)
        
    results.append(df)
    encoded_df = pd.concat(results).sort_index()

    return encoded_df

In [17]:
train_df = train_target_encoding(cols_to_encode)
test_df = test_target_encoding(test_df, cols_to_encode)

In [19]:
try:
    train_df.drop("id", axis=1, inplace=True)
    train_df.drop("HR_cut", axis=1, inplace=True)
    train_df.drop("cholesterol_cut", axis=1, inplace=True)
except KeyError:
    print("Columns have been removed already")

In [20]:
train_df.columns

Index(['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120',
       'EKG results', 'Max HR', 'Exercise angina', 'ST depression',
       'Slope of ST', 'Number of vessels fluro', 'Thallium', 'Heart Disease',
       'HR_group', 'cholesterol_group', 'reg_feature', 'ST_depression_group',
       'tar_Thallium', 'tar_Chest pain type', 'tar_Exercise angina',
       'tar_Number of vessels fluro', 'tar_Slope of ST', 'tar_HR_group',
       'tar_ST_depression_group'],
      dtype='object')

In [21]:
train_df["Thallium_7"] = np.where(train_df["Thallium"] == 7, 1, 0)
train_df["Chest_pain_type_4"] = np.where(train_df["Chest pain type"] == 4, 1, 0)
train_df["Male"] = np.where(train_df["Sex"] == 1, 1, 0)

train_df["Max HR_x_Thallium_7"] =               train_df["Max HR"] * train_df["Thallium_7"]
train_df["Male_x_Thallium_7"] =                 train_df["Male"] * train_df["Thallium_7"] 
train_df["Max HR_x_Chest pain type_4"] =        train_df["Max HR"] * train_df["Chest_pain_type_4"]
train_df["BP_x_Thallium_7"] =                   train_df["BP"] * train_df["Thallium_7"]
train_df["Cholesterol_x_Thallium_7"] =          train_df["Cholesterol"] * train_df["Thallium_7"]
train_df["Age_x_Thallium_7"] =                  train_df["Age"] * train_df["Thallium_7"]
train_df["Cholesterol_x_Chest pain type_4"] =   train_df["Cholesterol"] * train_df["Chest_pain_type_4"]
train_df["Male_x_Chest pain type_4"] =          train_df["Male"] * train_df["Chest_pain_type_4"]
train_df["BP_x_Chest pain type_4"] =            train_df["BP"] * train_df["Chest_pain_type_4"]
train_df["Age_x_Chest pain type_4"] =           train_df["Age"] * train_df["Chest_pain_type_4"]

In [22]:
test_df["Thallium_7"] = np.where(test_df["Thallium"] == 7, 1, 0)
test_df["Chest_pain_type_4"] = np.where(test_df["Chest pain type"] == 4, 1, 0)
test_df["Male"] = np.where(test_df["Sex"] == 1, 1, 0)

test_df["Max HR_x_Thallium_7"] =              test_df["Max HR"] * test_df["Thallium_7"]
test_df["Male_x_Thallium_7"] =                test_df["Male"] * test_df["Thallium_7"] 
test_df["Max HR_x_Chest pain type_4"] =       test_df["Max HR"] * test_df["Chest_pain_type_4"]
test_df["BP_x_Thallium_7"] =                  test_df["BP"] * test_df["Thallium_7"]
test_df["Cholesterol_x_Thallium_7"] =         test_df["Cholesterol"] * test_df["Thallium_7"]
test_df["Age_x_Thallium_7"] =                 test_df["Age"] * test_df["Thallium_7"]
test_df["Cholesterol_x_Chest pain type_4"] =  test_df["Cholesterol"] * test_df["Chest_pain_type_4"]
test_df["Male_x_Chest pain type_4"] =         test_df["Male"] * test_df["Chest_pain_type_4"]
test_df["BP_x_Chest pain type_4"] =           test_df["BP"] * test_df["Chest_pain_type_4"]
test_df["Age_x_Chest pain type_4"] =          test_df["Age"] * test_df["Chest_pain_type_4"]

In [35]:
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_leaf': [1, 2, 4, 10],
    'max_features': ['sqrt', 'log2', 0.3], # 0.3 means 30% of your features
    'max_samples': [0.5, 0.7, 0.8]        # Use only 70% of rows per tree
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(n_jobs=-1, random_state=1), 
    param_distributions=param_dist, 
    n_iter=10, 
    cv=3, 
    scoring='roc_auc', # Tune for AUC
    verbose=2,
    random_state=1
)


X = train_df.drop('Heart Disease', axis=1)
y = train_df['Heart Disease']

X_sub = X.sample(n=100000, random_state=1)
y_sub = y.loc[X_sub.index]

random_search.fit(X_sub, y_sub)

print(f"Best Parameters: {random_search.best_params_}")
best_model = random_search.best_estimator_

Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END max_depth=30, max_features=0.3, max_samples=0.5, min_samples_leaf=2, n_estimators=200; total time=   9.1s
[CV] END max_depth=30, max_features=0.3, max_samples=0.5, min_samples_leaf=2, n_estimators=200; total time=  12.2s
[CV] END max_depth=30, max_features=0.3, max_samples=0.5, min_samples_leaf=2, n_estimators=200; total time=  11.3s
[CV] END max_depth=None, max_features=log2, max_samples=0.5, min_samples_leaf=2, n_estimators=300; total time=   9.5s
[CV] END max_depth=None, max_features=log2, max_samples=0.5, min_samples_leaf=2, n_estimators=300; total time=  10.3s
[CV] END max_depth=None, max_features=log2, max_samples=0.5, min_samples_leaf=2, n_estimators=300; total time=  10.2s
[CV] END max_depth=None, max_features=log2, max_samples=0.5, min_samples_leaf=4, n_estimators=100; total time=   3.3s
[CV] END max_depth=None, max_features=log2, max_samples=0.5, min_samples_leaf=4, n_estimators=100; total time=   3.1s
[CV]

In [36]:
X = train_df.drop('Heart Disease', axis=1)
y = train_df['Heart Disease']

model = RandomForestClassifier(n_estimators=300, 
                               random_state=1,
                               min_samples_leaf=2,
                               max_samples=0.8,
                               max_features=0.3,
                               max_depth=10,
                               n_jobs=-1)
model.fit(X,y)

accuracy = model.score(X, y)
print(f"Training Accuracy: {accuracy:.2%}")

Training Accuracy: 88.92%


In [37]:
X_test = test_df[X.columns]

probabilities = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'id': test_df['id'],
    'Heart Disease': probabilities
})

submission.to_csv('submission3.csv', index=False)

In [40]:
from xgboost import XGBClassifier

X = train_df.drop('Heart Disease', axis=1)
y = train_df['Heart Disease']

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,           # XGBoost trees are usually shallower than RF
    learning_rate=0.1,      # The "step size" for improvements
    subsample=0.8,          # Equivalent to max_samples in RF
    colsample_bytree=0.3,   # Equivalent to max_features in RF
    tree_method='hist', 
    enable_categorical = True,
    random_state=1,
    n_jobs=-1
)

model.fit(X, y)

accuracy = model.score(X, y)
print(f"XGBoost Training Accuracy: {accuracy:.2%}")

XGBoost Training Accuracy: 89.20%


In [41]:
probabilities = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'id': test_df['id'],
    'Heart Disease': probabilities
})

submission.to_csv('submission4.csv', index=False)